In [ ]:
# RTX 5070 Ti fix — skip if PyTorch 2.7+cu128 already installed
import torch
if '2.7' not in torch.__version__:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q',
        'torch==2.7.0','torchvision','torchaudio',
        '--index-url','https://download.pytorch.org/whl/cu128'],
        capture_output=True)
    import IPython; IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print(f'PyTorch {torch.__version__} — sm_120 ready, skip install.')


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed}')
set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


In [ ]:
class Config:
    # ── Paths ────────────────────────────────────────────────────
    data_path    = '/home/user/Downloads/PEMS08.npz'
    adj_csv_path = '/home/user/Downloads/PEMS08.csv'
    num_nodes    = 170
    in_features  = 3
    seq_len      = 12   # 60 min history (12 × 5min)
    pred_len     = 12
    feature_idx  = 0
    noise_std    = 0.0
    train_ratio  = 0.7
    val_ratio    = 0.1

    # ── Model ────────────────────────────────────────────────────
    d_model    = 96     # residual channels
    d_skip     = 384    # skip channels
    d_end      = 256    # post-skip projection
    d_time     = 64     # temporal embedding dim
    n_layers   = 12     # WaveBlocks (3 dilation cycles of [1,2,4,8])
    kernel_size = 4
    adp_emb    = 20
    gcn_order  = 3
    n_supports = 3      # fwd + bwd + adaptive
    dropout    = 0.1

    # ── Multi-period fusion ──────────────────────────────────────
    n_heads_maf  = 4    # heads in MultiPeriodAttentionFusion

    # ── STFormer ─────────────────────────────────────────────────
    n_stf_layers = 2    # L = 2 spatial+temporal transformer layers
    n_heads_stf  = 4

    # ── Loss ─────────────────────────────────────────────────────
    huber_delta  = 0.2  # Huber threshold in normalised space

    # ── Training ─────────────────────────────────────────────────
    batch_size   = 48
    lr           = 1e-3
    warmup_eps   = 3
    epochs       = 150
    patience     = 30
    weight_decay = 1e-4
    best_path    = 'mdgwnet_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MD-GWNet v21 | d={cfg.d_model} | d_skip={cfg.d_skip} | layers={cfg.n_layers}')
print(f'  3-stream input (recent+hourly+daily) | STFormer L={cfg.n_stf_layers} | {device}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DATA — 3-stream dataset: recent | hourly (1h ago) | daily (24h ago)
# PEMS08 at 5-min intervals: 1h = 12 steps, 1 day = 288 steps
# ═══════════════════════════════════════════════════════════════════

def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np[None]) / std_np[None]
    feat_std_raw   = std_np[:, cfg.feature_idx].mean()
    norm_noise_std = cfg.noise_std / (feat_std_raw + 1e-8)
    import pandas as pd, os
    A_dist = None
    adj_csv = getattr(cfg, 'adj_csv_path', None)
    if adj_csv and os.path.exists(adj_csv):
        df_adj = pd.read_csv(adj_csv)
        A_raw  = np.zeros((N, N), dtype=np.float32)
        for _, row in df_adj.iterrows():
            i, j, c = int(row['from']), int(row['to']), float(row['cost'])
            if i < N and j < N:
                A_raw[i,j] = c; A_raw[j,i] = c
        nonzero = A_raw[A_raw > 0]
        sigma   = nonzero.std() if len(nonzero) > 0 else 1.0
        A_dist  = np.exp(-(A_raw**2) / (sigma**2 + 1e-8))
        np.fill_diagonal(A_dist, 0.0)
        A_dist  = A_dist / (A_dist.sum(1, keepdims=True) + 1e-8)
        print(f'Adjacency from CSV — {(A_dist>0).sum()} non-zero entries')
    else:
        for key in ('adjacency_matrix','adj_mx','adj'):
            if key in raw:
                A_dist = raw[key].astype(np.float32); break
        if A_dist is None:
            A_dist = np.eye(N, dtype=np.float32)
            print('WARNING: using identity adjacency')
    return data_norm, mean_np, std_np, A_dist, norm_noise_std


HOUR_OFFSET = 12   # 12 × 5min = 1 hour
DAY_OFFSET  = 288  # 288 × 5min = 24 hours

class TrafficDataset3Stream(Dataset):
    """
    Returns (x_rec, x_hour, x_day, y, tod, dow) per sample.
      x_rec : data[i : i+S]            — most recent S steps
      x_hour: data[i-12 : i-12+S]      — same window 1 hour ago
      x_day : data[i-288 : i-288+S]    — same window 1 day ago
    Requires i >= 288 (DAY_OFFSET).
    """
    def __init__(self, data, seq_len, pred_len, feature_idx,
                 noise_std=0.0, split_start=0, split_end=None, training=False):
        self.data      = data
        self.seq_len   = seq_len
        self.pred_len  = pred_len
        self.feat_idx  = feature_idx
        self.noise_std = noise_std
        self.training  = training
        T = len(data)
        split_end = split_end if split_end is not None else T
        # Enforce minimum start: need DAY_OFFSET lookback
        eff_start = max(split_start, DAY_OFFSET)
        last_i    = split_end - seq_len - pred_len + 1
        self.indices = list(range(eff_start, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        S, P = self.seq_len, self.pred_len
        rec  = self.data[i              : i + S             ].copy()
        hour = self.data[i-HOUR_OFFSET  : i-HOUR_OFFSET + S ].copy()
        day  = self.data[i-DAY_OFFSET   : i-DAY_OFFSET  + S ].copy()
        y    = self.data[i+S : i+S+P, :, self.feat_idx      ].copy()
        if self.training and self.noise_std > 0:
            rec  += np.random.randn(*rec.shape ).astype(np.float32) * self.noise_std
        tod  = np.array([(i+t) % 288       for t in range(S)], dtype=np.int64)
        dow  = np.array([((i+t)//288) % 7  for t in range(S)], dtype=np.int64)
        return (torch.from_numpy(rec),  torch.from_numpy(hour),
                torch.from_numpy(day),  torch.from_numpy(y),
                torch.from_numpy(tod),  torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data, mean_np, std_np, A_dist, norm_noise = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data=data, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx, noise_std=norm_noise)
    dl_tr = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=0,  split_end=t1, training=True),
                       shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=t1, split_end=t2, training=False),
                       shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=t2, split_end=T,  training=False),
                       shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist, norm_noise

print('3-stream dataset (recent + 1h ago + 24h ago) ready.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BUILDING BLOCKS
# ═══════════════════════════════════════════════════════════════════

class DiffusionGCN(nn.Module):
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.1):
        super().__init__()
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order
    def forward(self, x, supports):
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


class WaveBlock(nn.Module):
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.dilation    = dilation
        self.kernel_size = kernel_size
        conv_kw = dict(kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.filter_conv = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gate_conv   = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gcn         = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn          = nn.BatchNorm2d(d_model)
        self.drop        = nn.Dropout(dropout)
        self.skip_conv   = nn.Conv2d(d_model, d_skip,  (1,1))
        self.res_conv    = nn.Conv2d(d_model, d_model, (1,1))

    def forward(self, x, supports):
        residual = x
        pad = (self.kernel_size - 1) * self.dilation
        x_pad = F.pad(x, [pad, 0])
        f = torch.tanh   (self.filter_conv(x_pad))
        g = torch.sigmoid(self.gate_conv  (x_pad))
        x = self.drop(f * g)
        B, d, N, S = x.shape
        xg = x.permute(0,3,2,1).reshape(B*S, N, d)
        xg = self.gcn(xg, supports)
        x  = xg.reshape(B,S,N,d).permute(0,3,2,1)
        x  = self.bn(x)
        skip = self.skip_conv(x)
        x    = self.res_conv(x) + residual
        return x, skip


class MultiPeriodFusion(nn.Module):
    """
    Cross-attention fusion: recent (query) attends to hourly+daily (key/value).
    Adds context from 1h-ago and 24h-ago windows into the recent stream.
    """
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model * 2, d_model)  # hour+day concat
        self.v_proj   = nn.Linear(d_model * 2, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop     = nn.Dropout(dropout)
        self.norm     = nn.LayerNorm(d_model)

    def forward(self, h_rec, h_hour, h_day):
        # h_*: (B, d_model, N, S)
        B, d, N, S = h_rec.shape
        nh, dh     = self.n_heads, self.d_head
        def flatten(t): return t.permute(0,2,3,1).reshape(B*N, S, d)
        rec  = flatten(h_rec)   # (B*N, S, d)
        hour = flatten(h_hour)
        day  = flatten(h_day)
        kv   = torch.cat([hour, day], dim=-1)           # (B*N, S, 2d)
        Q    = self.q_proj(rec ).reshape(B*N, S, nh, dh).transpose(1,2)
        K    = self.k_proj(kv  ).reshape(B*N, S, nh, dh).transpose(1,2)
        V    = self.v_proj(kv  ).reshape(B*N, S, nh, dh).transpose(1,2)
        attn = F.softmax(Q @ K.transpose(-2,-1) / dh**0.5, dim=-1)
        attn = self.drop(attn)
        out  = (attn @ V).transpose(1,2).reshape(B*N, S, d)
        out  = self.out_proj(out)
        out  = self.norm(out + rec)                     # residual
        return out.reshape(B, N, S, d).permute(0,3,2,1)# (B, d, N, S)


class SpatialTransformer(nn.Module):
    """Multi-head self-attention across N nodes for each time step."""
    def __init__(self, d, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d, d*2), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d*2, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):
        B, d, N, S = x.shape
        xt = x.permute(0,3,2,1).reshape(B*S, N, d)     # (B*S, N, d)
        a, _ = self.attn(xt, xt, xt)
        xt = self.norm1(xt + a)
        xt = self.norm2(xt + self.ff(xt))
        return xt.reshape(B, S, N, d).permute(0,3,2,1)


class TemporalTransformer(nn.Module):
    """Multi-head self-attention across S time steps for each node."""
    def __init__(self, d, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d, d*2), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d*2, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):
        B, d, N, S = x.shape
        xn = x.permute(0,2,3,1).reshape(B*N, S, d)     # (B*N, S, d)
        a, _ = self.attn(xn, xn, xn)
        xn = self.norm1(xn + a)
        xn = self.norm2(xn + self.ff(xn))
        return xn.reshape(B, N, S, d).permute(0,3,2,1)


class STFormer(nn.Module):
    """L alternating Spatial + Temporal Transformer layers."""
    def __init__(self, d, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleList([
                SpatialTransformer(d, n_heads, dropout),
                TemporalTransformer(d, n_heads, dropout)
            ]) for _ in range(n_layers)
        ])

    def forward(self, x):
        for sp, tp in self.layers:
            x = tp(sp(x))
        return x

print('DiffusionGCN, WaveBlock, MultiPeriodFusion, STFormer defined.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MD-GWNet v21
# Architecture:
#   3 × start_conv  →  MultiPeriodFusion  →  WaveBlocks
#   →  skip aggregation  →  STFormer  →  output MLP
# ═══════════════════════════════════════════════════════════════════

class MDGWNet(nn.Module):
    def __init__(self, cfg, A_fwd, A_bwd):
        super().__init__()
        N, d = cfg.num_nodes, cfg.d_model

        # ── Input projections (one per period) ───────────────────
        self.sc_rec  = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_hour = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_day  = nn.Conv2d(cfg.in_features, d, (1,1))

        # ── Temporal embeddings (applied to recent stream) ───────
        self.emb_tod  = nn.Embedding(288, cfg.d_time)
        self.emb_dow  = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, d)

        # ── Node embedding (learnable per-node bias) ─────────────
        self.node_emb = nn.Parameter(torch.randn(1, d, N, 1) * 0.01)

        # ── Adaptive adjacency (self-learned graph) ──────────────
        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb))
        self.E2 = nn.Parameter(torch.randn(cfg.adp_emb, N))

        # ── Static adjacency (registered, not trained) ───────────
        self.register_buffer('A_fwd', A_fwd)
        self.register_buffer('A_bwd', A_bwd)

        # ── Multi-Period Attention Fusion ────────────────────────
        self.maf = MultiPeriodFusion(d, n_heads=cfg.n_heads_maf, dropout=cfg.dropout)

        # ── WaveBlocks ───────────────────────────────────────────
        dilations = [2**i for i in range(4)] * (cfg.n_layers // 4)
        self.blocks = nn.ModuleList([
            WaveBlock(d, cfg.d_skip, cfg.kernel_size, dil,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for dil in dilations
        ])

        # ── Post-skip projection ─────────────────────────────────
        self.skip_proj = nn.Conv2d(cfg.d_skip, cfg.d_end, (1,1))

        # ── Spatial-Temporal Transformer ─────────────────────────
        self.stformer = STFormer(cfg.d_end, n_layers=cfg.n_stf_layers,
                                 n_heads=cfg.n_heads_stf, dropout=cfg.dropout)

        # ── Output MLP ───────────────────────────────────────────
        self.out_conv = nn.Sequential(
            nn.Conv2d(cfg.d_end, cfg.d_end, (1,1)),
            nn.ReLU(),
            nn.Conv2d(cfg.d_end, cfg.pred_len, (1,1))
        )

    def _supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x_rec, x_hour, x_day, tod=None, dow=None):
        # x_*: (B, S, N, F)
        def to4d(x): return x.permute(0,3,2,1)  # → (B,F,N,S)

        h_rec  = self.sc_rec (to4d(x_rec ))   # (B,d,N,S)
        h_hour = self.sc_hour(to4d(x_hour))
        h_day  = self.sc_day (to4d(x_day ))

        h_rec  = h_rec + self.node_emb

        # Temporal embedding → inject into recent stream
        if tod is not None and dow is not None:
            te = self.time_proj(torch.cat([
                self.emb_tod(tod),   # (B,S,d_time)
                self.emb_dow(dow)
            ], dim=-1))              # (B,S,d)
            h_rec = h_rec + te.permute(0,2,1).unsqueeze(2)  # broadcast over N

        # Multi-period fusion: recent ← cross-attend(hourly, daily)
        x = self.maf(h_rec, h_hour, h_day)    # (B,d,N,S)

        # WaveBlocks
        sup = self._supports()
        skip_acc = 0
        for blk in self.blocks:
            x, skip = blk(x, sup)
            skip_acc = skip_acc + skip

        # Skip → project → STFormer
        h = F.relu(skip_acc)
        h = F.relu(self.skip_proj(h))          # (B,d_end,N,S)
        h = self.stformer(h)                   # (B,d_end,N,S)

        # Pool last 4 time steps
        h = h[..., -4:].mean(dim=-1, keepdim=True)  # (B,d_end,N,1)

        out = self.out_conv(h).squeeze(-1)     # (B,pred_len,N)
        return out


def build_model(cfg, A_dist_np, device):
    set_seed()
    sigma = A_dist_np[A_dist_np > 0].std() if (A_dist_np > 0).any() else 1.0
    A_fwd = torch.from_numpy(A_dist_np).to(device)
    A_bwd = torch.from_numpy(A_dist_np.T).to(device)
    # Normalise rows
    A_fwd = A_fwd / (A_fwd.sum(1, keepdim=True) + 1e-8)
    A_bwd = A_bwd / (A_bwd.sum(1, keepdim=True) + 1e-8)
    model = MDGWNet(cfg, A_fwd, A_bwd).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'MD-GWNet v21 ready | Parameters: {n_params:,}')
    print(f'  3-stream MAF + {cfg.n_layers}-layer WaveNet + STFormer(L={cfg.n_stf_layers})')
    # Smoke test
    with torch.no_grad():
        B = 2
        dummy = lambda: torch.randn(B, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
        tod = torch.randint(0, 288, (B, cfg.seq_len)).to(device)
        dow = torch.randint(0, 7,   (B, cfg.seq_len)).to(device)
        out = model(dummy(), dummy(), dummy(), tod, dow)
        ok  = list(out.shape) == [B, cfg.pred_len, cfg.num_nodes]
        print(f'  Output shape: {out.shape}  {chr(10003) if ok else chr(10007)+" CHECK!"}')
    return model

print('MDGWNet class defined.')


In [ ]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np, norm_noise = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)
print(f'Batches — Train:{len(dl_train)} | Val:{len(dl_val)} | Test:{len(dl_test)}')

model = build_model(cfg, A_dist_np, device)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS  (Huber for training, MAE/RMSE/MAPE for eval)
# ═══════════════════════════════════════════════════════════════════

def masked_huber(pred, target, delta=0.2, mask_val=0.0):
    """Huber loss — robust to outliers; transitions L2→L1 at delta."""
    mask = (target.abs() > mask_val).float()
    diff = torch.abs(pred - target)
    loss = torch.where(diff <= delta,
                       0.5 * diff**2,
                       delta * (diff - 0.5 * delta))
    return (loss * mask).sum() / (mask.sum() + 1e-8)

def masked_mae(pred, target, mask_val=0.0):
    mask = (target.abs() > mask_val).float()
    return (torch.abs(pred - target) * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, target, mask_val=0.0):
    mask = (target.abs() > mask_val).float()
    return ((((pred-target)**2) * mask).sum() / (mask.sum() + 1e-8)).sqrt()

def masked_mape(pred, target, mask_val=10.0):
    mask = (target.abs() > mask_val).float()
    return (torch.abs((pred-target)/(target.abs()+1.0)) * mask).sum() / (mask.sum()+1e-8) * 100


scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);  x_hour = x_hour.to(device)
        x_day  = x_day.to(device);  y      = y.to(device)
        tod    = tod.to(device);     dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
            loss = masked_huber(pred, y, delta=cfg.huber_delta)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);  x_hour = x_hour.to(device)
        x_day  = x_day.to(device);  y      = y.to(device)
        tod    = tod.to(device);     dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Huber loss + train/eval defined.')


In [ ]:
set_seed()

optimizer = torch.optim.AdamW(model.parameters(),
                               lr=cfg.lr, weight_decay=cfg.weight_decay)

def lr_lambda(ep):
    return (ep+1) / cfg.warmup_eps if ep < cfg.warmup_eps else 1.0

warmup_sched  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=100, T_mult=1, eta_min=1e-5)
plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=15, factor=0.6, min_lr=1e-5)

best_val_mae = float('inf')
patience_cnt = 0
history = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

print(f'Training MD-GWNet v21 | Huber(delta={cfg.huber_delta}) | 3-stream + STFormer')
print(f'Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*70)

for epoch in range(1, cfg.epochs+1):
    tr_loss = train_epoch(model, dl_train, optimizer, device)
    val_mae, val_rmse, val_mape = eval_epoch(model, dl_val, device, mean_flow, std_flow)

    if epoch <= cfg.warmup_eps:
        warmup_sched.step()
    else:
        cosine_sched.step(epoch - cfg.warmup_eps)
    plateau_sched.step(val_mae)

    history['train_loss'].append(tr_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stop at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = ' *** BEAT BASELINE!' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={tr_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val MAE: {best_val_mae:.3f}  (baseline: 13.114)')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss (Huber)')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig('training_curves_v21.png', dpi=150); plt.show()


In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);  x_hour = x_hour.to(device)
        x_day  = x_day.to(device)
        tod    = tod.to(device);     dow    = dow.to(device)
        pred   = model(x_rec, x_hour, x_day, tod, dow)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu()); all_true.append(y_d.cpu())
    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = torch.abs(P-Y).mean().item()
    rmse = ((P-Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100
    print('\n' + '='*55)
    print('  TEST RESULTS  —  averaged over all 12 horizons')
    print('='*55)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Delta={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Delta={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Delta={mape-8.471:+.3f}%')
    print('='*55)
    return mae, rmse, mape

paper_eval(model, dl_test, device, mean_flow, std_flow)


In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);  x_hour = x_hour.to(device)
        x_day  = x_day.to(device)
        tod    = tod.to(device);     dow    = dow.to(device)
        pred   = model(x_rec, x_hour, x_day, tod, dow)
        pred_d = pred * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.to(device) * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:],y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:],y_d[:,h,:]).item())
    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*50)
    for h, lbl in zip([2,5,11],['3-step(15min)','6-step(30min)','12-step(60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, device, mean_flow, std_flow)
